In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/pune/bus")

routes = pd.read_csv(data_path / "routes.txt", dtype=str)
trips = pd.read_csv(data_path / "trips.txt", dtype=str)
stops = pd.read_csv(data_path / "stops.txt", dtype=str)
stop_times = pd.read_csv(data_path / "stop_times.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)
print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)
print("\nStop times columns:", stop_times.columns.tolist())

Routes: (628, 5)
Trips: (10128, 6)
Stops: (6745, 4)
Stop times: (429302, 6)

Stop times columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'timepoint']


In [2]:
trip_counts = trips.groupby("route_id")["trip_id"].nunique().reset_index(name="total_trips")

trip_stops = trips[["route_id", "trip_id"]].merge(
    stop_times[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

stop_counts = trip_stops.groupby("route_id")["stop_id"].nunique().reset_index(name="total_stops")

route_features = routes.merge(trip_counts, on="route_id", how="left")
route_features = route_features.merge(stop_counts, on="route_id", how="left")

print(route_features.head())

  route_id agency_id route_short_name  \
0        6     PMPML              100   
1        7     PMPML              100   
2       10     PMPML              103   
3       14     PMPML              104   
4       16     PMPML              107   

                                     route_long_name route_type  total_trips  \
0  Hinjawadi Maan Phase 3 to Ma Na Pa Dengle Pul ...          3           36   
1  Ma Na Pa Dengle Pul Nadikhathi to Hinjawadi Ma...          3           43   
2                       Kothrud Depot to Katraj (UP)          3           29   
3                   Dsk Vishwa to Shivaji Nagar (UP)          3           11   
4                  Pimplegurav to Ganpati Matha (UP)          3            3   

   total_stops  
0           52  
1           51  
2           35  
3           44  
4           57  


In [3]:
stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="coerce"
)

def gtfs_hour(t):
    try:
        h, m, s = map(int, t.split(":"))
        return h + m / 60 + s / 3600
    except:
        return None

stop_times["arrival_hour"] = stop_times["arrival_time"].apply(gtfs_hour)
stop_times["departure_hour"] = stop_times["departure_time"].apply(gtfs_hour)

def time_period(h):
    if pd.isna(h):
        return "Unknown"
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

stop_times["arrival_period"] = stop_times["arrival_hour"].apply(time_period)
stop_times["departure_period"] = stop_times["departure_hour"].apply(time_period)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

first_departures["departure_service_hour"] = (
    first_departures["departure_hour"].apply(
        lambda h: int(h) if pd.notna(h) else None
    )
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

print(stop_times[["arrival_hour", "departure_hour", "departure_period"]].head())
print("\nHourly trips:")
print(hourly_trips.head())

   arrival_hour  departure_hour departure_period
0      8.500000        8.500000          Morning
1      8.516667        8.516667          Morning
2      8.552500        8.552500          Morning
3      8.569167        8.569167          Morning
4      8.585833        8.585833          Morning

Hourly trips:
  route_id  departure_service_hour  trip_count
0       10                       5           1
1       10                       6           1
2       10                       7           4
3       10                       8           4
4       10                       9           3


In [4]:
output_path = Path("../data/processed/pune")
output_path.mkdir(parents=True, exist_ok=True)

route_features.to_csv(output_path / "route_profile.csv", index=False)

hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)

stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Pune feature files saved successfully!")

Pune feature files saved successfully!


In [5]:
print("Missing values:")
print(route_features.isnull().sum())

print("\nDuplicate route IDs:", routes["route_id"].duplicated().sum())
print("Duplicate trip IDs:", trips["trip_id"].duplicated().sum())
print("Missing arrival hours:", stop_times["arrival_hour"].isnull().sum())
print("Missing departure hours:", stop_times["departure_hour"].isnull().sum())

Missing values:
route_id            0
agency_id           0
route_short_name    0
route_long_name     0
route_type          0
total_trips         0
total_stops         0
dtype: int64

Duplicate route IDs: 0
Duplicate trip IDs: 0
Missing arrival hours: 0
Missing departure hours: 0


In [6]:
def gtfs_minute(t):
    try:
        h, m, s = map(int, t.split(":"))
        return m
    except:
        return None

stop_times["arrival_minute"] = stop_times["arrival_time"].apply(gtfs_minute)
stop_times["departure_minute"] = stop_times["departure_time"].apply(gtfs_minute)

stop_times.to_csv(output_path / "time_features.csv", index=False)

print(stop_times[[
    "arrival_hour", "arrival_minute",
    "departure_hour", "departure_minute"
]].head())

   arrival_hour  arrival_minute  departure_hour  departure_minute
0      8.500000              30        8.500000                30
1      8.516667              31        8.516667                31
2      8.552500              33        8.552500                33
3      8.569167              34        8.569167                34
4      8.585833              35        8.585833                35


In [7]:
features = pd.DataFrame({
    "Feature": [
        "route_id",
        "total_trips",
        "total_stops",
        "arrival_hour",
        "arrival_minute",
        "departure_hour",
        "departure_minute",
        "arrival_period",
        "departure_period",
        "departure_service_hour",
        "trip_count"
    ],
    "Description": [
        "Unique route identifier",
        "Number of distinct trips on a route",
        "Number of distinct stops served by a route",
        "Scheduled arrival time in decimal hours",
        "Arrival minute",
        "Scheduled departure time in decimal hours",
        "Departure minute",
        "Arrival time category",
        "Departure time category",
        "Integer service-hour bucket",
        "Number of distinct trips in that hour"
    ],
    "Data_Type": [
        "Categorical", "Numerical", "Numerical",
        "Numerical", "Numerical", "Numerical",
        "Numerical", "Categorical", "Categorical",
        "Numerical", "Numerical"
    ]
})

features.to_csv("../reports/feature_dictionary.csv", index=False)

print("Feature dictionary saved!")

Feature dictionary saved!
